# Activity 1 — RAG with and without filter

This is Activity 1 of the LLM Observability Hackathon. Run the cells top-to-bottom in a fresh kernel.

## Purpose
Compare a RAG answer produced with unfiltered retrieval against one produced with a tier filter. The trace records the retriever's filter, the retrieved document, and the generation input and output.

## What to expect
The unfiltered run can retrieve a policy from any tier; in this run it retrieves the Gold policy. The filtered run retrieves a policy from the requested tier, the Basic policy. Compare the two answers and the retrieved documents. The notebook creates the traces `activity-1-rag-no-filter` and `activity-1-rag-filter`.

## Bad practice
Retrieve the top document by text similarity alone and pass it to the model without checking the customer's entitlement.

## Good practice
Filter the candidate documents by the customer's tier before ranking, then pass the selected policy to the model explicitly.

## Why the good practice works
The model cannot infer the customer's tier from the question alone. Filtering at retrieval keeps a policy for another tier out of the generation input, and the trace shows which filter produced the document.

## Deliverable
The two trace URLs and a comparison of the retrieved documents and answers.


In [1]:
import time
from IPython.display import Markdown, display
from experiment_utils import (client, langfuse, MODEL, RUN_ID, print_summary,
                              propagate_attributes, record_trace, retrieve)


In [2]:
def run_rag(customer_message, tier, variant):
    trace_name = f"activity-1-rag-{variant}"
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-1-{RUN_ID}",
            tags=["activity-1", variant],
            trace_name=trace_name,
        ):
            doc = retrieve("checked bag allowance", tier)
            with langfuse.start_as_current_observation(
                name="activity-1-retrieval",
                as_type="retriever",
                input={"query": "checked bag allowance",
                       "filter": None if tier is None else {"tier": tier}},
                output={"doc_id": doc["id"], "tier": doc["tier"], "text": doc["text"]},
            ):
                pass
            messages = [
                {"role": "system", "content": "Answer using the retrieved policy document."},
                {"role": "user", "content": f"{customer_message}\n\nRetrieved policy:\n{doc['text']}"},
            ]
            started = time.time()
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=300,
                name="activity-1-generation",
            )
            latency = time.time() - started
            answer = response.choices[0].message.content
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"answer": answer, "doc_id": doc["id"]})
    usage = response.usage
    return {
        "variant": variant,
        "answer": answer,
        "doc": doc,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "latency": latency,
        "trace_id": trace_id,
        "trace_url": record_trace(f"activity-1-{variant}", trace_id),
    }

customer = "How many checked bags do I get on my booking?"
runs = [run_rag(customer, None, "no-filter"), run_rag(customer, "basic", "filter")]

rows = ["| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |",
        "|---|---|---|---|---|---|---|"]
for run in runs:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {doc['id']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |
|---|---|---|---|---|---|---|
| no-filter | baggage-gold (gold) | You get **one 23 kg checked bag** included with the Gold fare. A second checked bag costs **60 EUR**. | 72 | 97 | 1.7 | http://localhost:3000/project/langfuse-observability/traces/05f3c8cace914ce28b7c70b7e09ab1dc |
| filter | baggage-basic (basic) | You don’t get any checked bags included with the Basic fare. You can add a 23 kg checked bag for 45 EUR. | 82 | 103 | 1.6 | http://localhost:3000/project/langfuse-observability/traces/cdaf3c31477a38dccf4cb4ed360aaa9e |

## Wrap-up

Run this cell to print the session id and trace URLs for this activity, then flush the Langfuse client. Submit the trace URLs and your findings.


In [3]:
print_summary(1)
langfuse.flush()
print("flushed")

run id: 077ed5d7
session activity-1-077ed5d7
activity-1-no-filter: http://localhost:3000/project/langfuse-observability/traces/05f3c8cace914ce28b7c70b7e09ab1dc
activity-1-filter: http://localhost:3000/project/langfuse-observability/traces/cdaf3c31477a38dccf4cb4ed360aaa9e
flushed
